In [1]:
%%time
import pandas as pd
import numpy as np
import requests
from lxml.html import fromstring
# Verify the URLs and only store the valid ones in a list
data_url = "https://edge.pse.com.ph/companyPage/stockData.do?cmpy_id="
mngt_url = "https://edge.pse.com.ph/companyPage/directors_and_management_list.do?cmpy_id="
stock_num = []

Wall time: 341 ms


In [2]:
%%time
stock_dict = {}
from bs4 import BeautifulSoup
for i in range(0,301,1):
    num = str(i)
    r = requests.get(data_url+num)
    title = fromstring(r.content).findtext('.//title')
    if title == 'Stock Data':
        soup = BeautifulSoup(r.content, 'html.parser')
        symbol = soup.find('option').text
        stock_dict[symbol] = num
        #print(list(stock_dict.keys())[i],list(stock_dict.values())[i])

Wall time: 1min 28s


In [4]:
%%time
network_list = []
for i,j in stock_dict.items():
    mngt_table = pd.read_html('https://edge.pse.com.ph/companyPage/directors_and_management_list.do?cmpy_id='+j)
    for k in range(0,len(mngt_table[0])):
        network_list.append([i+" ",mngt_table[0].iloc[k,1],mngt_table[0].iloc[k,0]])


Wall time: 8min 3s


In [5]:
network_list[:10]

[['MB ', 'Basilio C. Yap', 'Chairman of the Board'],
 ['MB ', 'Emilio C. Yap III', 'Vice Chairman'],
 ['MB ', 'Hilario G. Davide, Jr.', 'Vice Chairman and Independent Director'],
 ['MB ', 'Alberto G. Romulo', 'Vice Chairman and Independent Director'],
 ['MB ', 'Enrique Y. Yap, Jr.', 'Director'],
 ['MB ', 'Francis Y. Gaw', 'Director'],
 ['MB ', 'Crispulo J. Icban, Jr.', 'Director'],
 ['MB ', 'Benjamin C. Yap', 'Director'],
 ['MB ', 'Maria Georgina Perez-De Venecia', 'Independent Director'],
 ['MB ', 'Armando L. Suratos', 'Independent Director']]

In [31]:
import numpy as np
network_list = np.array(network_list)

In [35]:
title = list(set(network_list[:,2]))

In [38]:
weight = [8,4,6,7,6,5,1,5,1,6,7,5,8,4,7,7,1,3,6,8,3,5,1,1,5,5,7,6]
lookup_table = dict(zip(title,weight))

In [40]:
network_list[:,2]

array(['Chairman of the Board', 'Vice Chairman',
       'Vice Chairman and Independent Director', ...,
       'Independent Director', 'Independent Director',
       'Independent Director'], dtype='<U46')

In [43]:
                                                                                                                       
position_weight = [lookup_table.get(x, x) for x in network_list[:,2]]                                                                                                                    

network_list[:,2] = position_weight   

In [44]:
network_list

array([['MB ', 'Basilio C. Yap', '7'],
       ['MB ', 'Emilio C. Yap III', '5'],
       ['MB ', 'Hilario G. Davide, Jr.', '6'],
       ...,
       ['MWC ', 'Jaime C. Laya', '1'],
       ['MWC ', 'Sherisa P. Nuesa', '1'],
       ['MWC ', 'Jose L. Cuisia, Jr.', '1']], dtype='<U46')

In [47]:
table = pd. DataFrame(network_list, columns=['Ticker', 'Name','Weight'])

In [49]:
table.to_csv('mafia.csv',index=False)

In [66]:
from pyvis.network import Network
got_net = Network(height='100%', width='75%', bgcolor='#222222', font_color='white')

# set the physics layout of the network

got_data = pd.read_csv('mafia.csv')

sources = got_data['Ticker']
targets = got_data['Name']
weights = got_data['Weight']

edge_data = zip(sources, targets, weights)

for e in edge_data:
    src = e[0]
    dst = e[1]
    w = e[2]

    got_net.add_node(src, src, title=src,color = "yellow")
    got_net.add_node(dst, dst, title=dst)
    #got_net.add_edge(src, dst, value=w)
    got_net.add_edge(src, dst, )

neighbor_map = got_net.get_adj_list()

# add neighbor data to node hover data
for node in got_net.nodes:
    node['title'] += ' Neighbors:<br>' + '<br>'.join(neighbor_map[node['id']])
    node['value'] = len(neighbor_map[node['id']])
got_net.show_buttons(filter_=['physics'])
#got_net.set_options(
#    """
 #   var options = {"physics": {"forceAtlas2Based": 
  #  {"gravitationalConstant": -298,"springLength": 0},
   # "maxVelocity": 150,"minVelocity": 14,"solver": "forceAtlas2Based"}}
    #"""
#)
got_net.show('mafia.html')

In [ ]:
mafia_list=[]
for l in network_list:
    n = []
    for e in l:
        e = e.replace(',', '')
        n.append(e)
    mafia_list.append(n)

In [ ]:
mafia_list[:10]

In [ ]:
import csv
np.savetxt("mafia.csv", mafia_list, delimiter=',',fmt='%s')

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
#import dzcnapy_plotlib as dzcnapy
import csv
with open("mafia.csv") as infile:
    csv_reader = csv.reader(infile)
    G = nx.Graph(csv_reader)
d = dict(G.degree)


In [ ]:
#nx.circular_layout(G)
#nx.draw_random(G)  
#nx.draw_circular(G)  
#nx.draw_spectral(G)  
#nx.draw_spring(G) 

plt.figure(3,figsize=(20,12))
#nx.draw_networkx(G,with_labels=True,node_size=60,font_size=8,nodelist=d.keys())
nx.draw(G, with_labels=True,nodelist=d.keys(), node_size=[v * 100 for v in d.values()])
plt.show()

In [ ]:
from pyvis.network import Network
nt = Network(height='100%', width='70%', bgcolor='#222222', font_color='white')
# populates the nodes and edges data structures
nt.show_buttons(filter_=['physics'])
#nt.show_buttons()
#nt.enable_physics(True)
nt.from_nx(G)
nt.show('nx.html')